# Almost Ramannet version from the orginal paper 

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.linear_model import LogisticRegression
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), '..'))


from config import STRAINS, ATCC_GROUPINGS, antibiotics

def set_seed(seed=123):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), '..', 'data')
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Load the dataset
X_ref = np.load(os.path.join(DATA_DIR, "X_reference.npy"))
y_ref = np.load(os.path.join(DATA_DIR, "y_reference.npy")).astype(int)
X_te = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_te = np.load(os.path.join(DATA_DIR, "y_test.npy")).astype(int)

print("Dataset shapes:")
print(f"X_reference: {X_ref.shape}")
print(f"y_reference: {y_ref.shape}")
print(f"X_test: {X_te.shape}")
print(f"y_test: {y_te.shape}")

# Check strain distribution
print(f"\nStrain distribution in reference set:")
strain_counts = Counter(y_ref)
for strain_id, count in sorted(strain_counts.items()):
    print(f"Strain {strain_id} ({STRAINS[strain_id]}): {count} samples")

In [ ]:
def get_binary_strains():
    """Get strain IDs that belong to penicillin or Meropenem classes"""
    binary_strains = []
    for strain_id in range(30):  # All possible strain IDs
        antibiotic_id = ATCC_GROUPINGS[strain_id]
        if antibiotic_id == 0 or antibiotic_id == 5:  # Meropenem or Peniciline
            binary_strains.append(strain_id)
    return sorted(binary_strains)

# Get focused strains for detailed mapping
focused_strains = get_binary_strains()
print("\nTRUE labels: strain -> antibiotic mapping")

for sid in focused_strains:
    true_ab = ATCC_GROUPINGS[sid]
    sname = STRAINS[sid]
    print(f"{sid:2d} | {sname:<20} : [{true_ab}] {antibiotics[true_ab]}")

In [ ]:
# Get the focused strains (penicillin and meropenem only)
focused_strains = get_binary_strains()

# Separate into penicillin and meropenem groups
penicillin_strains = [s for s in focused_strains if ATCC_GROUPINGS[s] == 5]
meropenem_strains = [s for s in focused_strains if ATCC_GROUPINGS[s] == 0]

# Create reordering indices - penicillin first, then meropenem
reorder_indices = []
for strain_id in penicillin_strains:
    # Find all indices where y_te equals this strain_id
    strain_indices = np.where(y_te == strain_id)[0]
    reorder_indices.extend(strain_indices)

for strain_id in meropenem_strains:
    # Find all indices where y_te equals this strain_id
    strain_indices = np.where(y_te == strain_id)[0]
    reorder_indices.extend(strain_indices)

# Reorder the data according to focused strains
X_te_reordered = X_te[reorder_indices]
y_te_reordered = y_te[reorder_indices]

# Get antibiotic labels for the reordered data
y_te_ab_reordered = np.array([ATCC_GROUPINGS[strain_id] for strain_id in y_te_reordered])

# Plot the focused spectra as a heatmap
plt.figure(figsize=(15, 8))
plt.imshow(X_te_reordered, cmap='viridis', aspect='auto', interpolation='nearest')
plt.title('Penicillin and Meropenem Test Spectra - Grouped by Antibiotic Class')
plt.xlabel('Wavenumber Index (1000 points)')
plt.ylabel('Spectrum Index - Penicillin first, then Meropenem')

# Add y-axis labels showing strain and antibiotic info
yticks = np.arange(0, len(y_te_ab_reordered), 100)
ytick_labels = []
for i in yticks:
    strain_id = y_te_reordered[i]
    ab_id = y_te_ab_reordered[i]
    strain_name = STRAINS[strain_id]
    ab_name = antibiotics[ab_id]
    ytick_labels.append(f'{i}: {strain_id} ({strain_name}) - {ab_name}')

plt.yticks(yticks, ytick_labels, rotation=0, fontsize=8)

plt.colorbar(label='Intensity')
plt.tight_layout()
plt.show()

# Print the reordering information
print("Focused strains (Penicillin and Meropenem only):")
print("Penicillin strains =", penicillin_strains)
print("Meropenem strains =", meropenem_strains)
print(f"Total spectra in focused set: {len(reorder_indices)}")
print("First 10 strain IDs in new order:", y_te_reordered[:10])
print("First 10 antibiotic classes in new order:", y_te_ab_reordered[:10])

# Show strain distribution
print("\nStrain distribution in focused test set:")
from collections import Counter
strain_counts = Counter(y_te_reordered)
for strain_id, count in sorted(strain_counts.items()):
    ab_id = ATCC_GROUPINGS[strain_id]
    print(f"Strain {strain_id} ({STRAINS[strain_id]}) - {antibiotics[ab_id]}: {count} samples")

In [ ]:
STRAIN_ORDER = np.arange(30)  
sid_ref_all = np.repeat(STRAIN_ORDER, 2000)
sid_te_all  = np.repeat(STRAIN_ORDER, 100)

focused_strains = get_binary_strains()
ref_mask = np.isin(sid_ref_all, focused_strains)
te_mask  = np.isin(sid_te_all,  focused_strains)

sid_ref_filtered = sid_ref_all[ref_mask]
sid_te_filtered  = sid_te_all[te_mask]


X_ref_filtered = X_ref[ref_mask]
X_te_filtered  = X_te[te_mask]

CLASS_TO_BIN = {0: 0, 5: 1}
y_ref_filtered = np.array([CLASS_TO_BIN[ATCC_GROUPINGS[s]] for s in sid_ref_filtered], dtype=int)
y_te_filtered  = np.array([CLASS_TO_BIN[ATCC_GROUPINGS[s]] for s in sid_te_filtered],  dtype=int)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RamanNetResidualBlock(nn.Module):
    """
    Residual block for RamanNet architecture as described in the paper.
    Each residual layer contains a shortcut connection between input and output
    of one convolutional layer.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(RamanNetResidualBlock, self).__init__()
        
        # Single convolutional layer as per paper description
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, 
                             padding=kernel_size//2, bias=False)
        self.bn = nn.BatchNorm1d(out_channels)
        
        # Shortcut connection
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn(self.conv(x)))
        # FIXED: Use out = out + self.shortcut(x) instead of += to avoid in-place operation
        out = out + self.shortcut(x)
        out = F.relu(out)
        return out


class BinaryClassifier(nn.Module):
    """
    BinaryClassifier model following the exact RamanNet architecture described in the paper.
    
    Architecture:
    1. Initial convolution layer (kernel size 7)
    2. Two residual layers (kernel sizes 7 and 3)
    3. Flatten layer
    4. Final fully connected classification layer (2 classes for binary classification)
    
    All convolution kernel numbers are set to 1 as specified in the paper.
    Uses kaiming normal initialization.
    """
    def __init__(self, input_dim=1000, n_classes=2):
        super(BinaryClassifier, self).__init__()
        self.input_dim = input_dim
        self.n_classes = n_classes
        
        # Initial convolution layer (kernel size 7, 1 channel)
        self.conv1 = nn.Conv1d(1, 1, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(1)
        
        # First residual layer (kernel size 7)
        self.residual1 = RamanNetResidualBlock(1, 1, kernel_size=7)
        
        # Second residual layer (kernel size 3)
        self.residual2 = RamanNetResidualBlock(1, 1, kernel_size=3)
        
        # Flatten layer
        self.flatten = nn.Flatten()
        
        # Final fully connected classification layer
        self.classifier = nn.Linear(input_dim, n_classes)
        
        # Initialize weights using kaiming normal initialization
        self._initialize_weights()

    def forward(self, x):
        # Initial convolution layer
        x = F.relu(self.bn1(self.conv1(x)))
        
        # Two residual layers
        x = self.residual1(x)
        x = self.residual2(x)
        
        # Flatten
        x = self.flatten(x)
        
        # Final classification layer
        x = self.classifier(x)
        
        return x

    def _initialize_weights(self):
        """
        Initialize weights using kaiming normal initialization as specified in the paper.
        """
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.constant_(m.bias, 0)

In [ ]:
def prepare_data(X, y, batch_size=32, shuffle=True):
    """Convert numpy arrays to DataLoader with shape [B,1,L]."""
    X_tensor = torch.as_tensor(X, dtype=torch.float32).unsqueeze(1)  # [N,1,L]
    y_tensor = torch.as_tensor(y, dtype=torch.long)
    dataset  = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad(set_to_none=True)
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        # sample-weighted loss to avoid bias from a short last batch
        bs = target.size(0)
        total_loss += loss.detach().item() * bs
        pred = output.detach().argmax(dim=1)
        correct += (pred == target).sum().item()
        total += bs

    avg_loss = total_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc

In [ ]:
# Evaluation function
def evaluate(model, test_loader, criterion, device):
    model.eval()
    assert not model.training, "Model must be in eval() during evaluation."

    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            bs = target.size(0)
            total_loss += loss.detach().item() * bs
            pred = output.detach().argmax(dim=1)

            correct += (pred == target).sum().item()
            total += bs
            all_preds.extend(pred.cpu().tolist())
            all_targets.extend(target.cpu().tolist())

    avg_loss = total_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc, all_preds, all_targets

In [ ]:
def train_eval_permutation_with_epochs(
    X_train, y_train, X_test, y_test,
    epochs=10, lr=5e-5, batch_size=256, weight_decay=1e-2
):
    
    torch.manual_seed(123)
    torch.cuda.manual_seed_all(123)
    
    # Infer spectrum length from data
    input_dim = X_train.shape[1]
    assert X_test.shape[1] == input_dim, "Train/Test spectrum length mismatch."

    # Model (RamanNet) + loaders
    model = BinaryClassifier(input_dim=input_dim, n_classes=2).to(DEVICE)
    train_loader = prepare_data(X_train, y_train, batch_size=batch_size, shuffle=True)
    test_loader  = prepare_data(X_test,  y_test,  batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_test_acc = -1.0
    best_epoch = -1
    best_state = None

    for epoch in range(1, epochs + 1):
        # ---- Train ----
        model.train()
        running_correct, running_total, running_loss = 0, 0, 0.0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)  # [B,1,L]
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.detach().item() * xb.size(0)
            preds = logits.argmax(dim=1)
            running_correct += (preds == yb).sum().item()
            running_total += xb.size(0)

        train_acc = 100.0 * running_correct / max(1, running_total)

        # ---- Eval ----
        model.eval()
        test_correct, test_total, test_loss_sum = 0, 0, 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)

                logits = model(xb)
                loss = criterion(logits, yb)

                test_loss_sum += float(loss) * xb.size(0)
                preds = logits.argmax(dim=1)
                test_correct += (preds == yb).sum().item()
                test_total += xb.size(0)

        test_acc = 100.0 * test_correct / max(1, test_total)
        print(f"    Epoch {epoch}: Train Acc = {train_acc:.2f}%, Test Acc = {test_acc:.2f}%")

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"[BEST] Epoch {best_epoch}: Test Acc = {best_test_acc:.2f}%")
    return best_test_acc / 100.0

In [ ]:
# --- binary class mapping (Meropenem=0 -> 0, Penicillin=5 -> 1)
CLASS_TO_BIN = {0: 0, 5: 1}
BIN_TO_CLASS = {0: 0, 1: 5}

LAST_PERM_MAP = None

def _make_strain_perm_map(focused_strains, rng):
    """
    Build ONE strain->binary map per permutation, preserving
    the number of Meropenem (0) and Penicillin (5) strains.
    """
    n_mer = sum(ATCC_GROUPINGS[s] == 0 for s in focused_strains)  # Meropenem
    n_pen = sum(ATCC_GROUPINGS[s] == 5 for s in focused_strains)  # Penicillin

    # build a label list of the correct total length, then shuffle
    perm_bins = np.array([CLASS_TO_BIN[0]] * n_mer + [CLASS_TO_BIN[5]] * n_pen, dtype=int)
    rng.shuffle(perm_bins)

    # safety
    assert len(perm_bins) == len(focused_strains), \
        f"perm_bins({len(perm_bins)}) != focused_strains({len(focused_strains)})"

    # assign one permuted binary label per strain
    return {s: perm_bins[i] for i, s in enumerate(focused_strains)}


def print_permutation_mapping(perm_idx, y_ref_perm, y_te_perm):
    print(f"\nPermutation {perm_idx}: strain -> antibiotic (TRUE -> PERMUTED)")
    if LAST_PERM_MAP is None:
        print("  [warn] LAST_PERM_MAP is not set")
        return
    for sid in focused_strains:
        true_ab = ATCC_GROUPINGS[sid]             
        perm_bin = LAST_PERM_MAP[sid]             
        perm_ab  = BIN_TO_CLASS[perm_bin]         
        sname = STRAINS[sid]
        print(f"{sid:2d} | {sname:<20} : [{true_ab}] {antibiotics[true_ab]}  ->  [{perm_ab}] {antibiotics[perm_ab]}")


# Permutation test parameters
N_PERMUTATIONS = 1000  # Number of permutations
PERM_EPOCHS = 10   # Epochs for each permutation
RANDOM_SEED = 42

In [ ]:
# Do true training with epoch-by-epoch progress
print(f"\nTRUE labels training:")
true_accuracy = train_eval_permutation_with_epochs(
    X_ref_filtered, y_ref_filtered,
    X_te_filtered, y_te_filtered,
    epochs=PERM_EPOCHS
)
print(f"TRUE labels final accuracy: {true_accuracy:.2%}")

In [ ]:
import itertools

def _make_unique_strain_perm_maps(focused_strains, n_permutations=1000, seed=42):
    """
    Generate n_permutations unique strain->binary maps, preserving
    the number of Meropenem (0) and Penicillin (5) strains.
    """
    n_mer = sum(ATCC_GROUPINGS[s] == 0 for s in focused_strains)  # Meropenem
    n_pen = sum(ATCC_GROUPINGS[s] == 5 for s in focused_strains)  # Penicillin
    
    # Create the base label list
    base_labels = [CLASS_TO_BIN[0]] * n_mer + [CLASS_TO_BIN[5]] * n_pen
    
    # Generate all possible unique permutations
    unique_perms = set()
    rng = np.random.default_rng(seed)
    
    # Keep generating until we have enough unique permutations
    while len(unique_perms) < n_permutations:
        # Shuffle the base labels
        perm_labels = base_labels.copy()
        rng.shuffle(perm_labels)
        
        # Convert to tuple for hashing
        perm_tuple = tuple(perm_labels)
        unique_perms.add(perm_tuple)
    
    # Convert back to list of dictionaries
    perm_maps = []
    for perm_tuple in list(unique_perms)[:n_permutations]:
        perm_map = {s: perm_tuple[i] for i, s in enumerate(focused_strains)}
        perm_maps.append(perm_map)
    
    return perm_maps

def _make_unique_strain_perm_maps_itertools(focused_strains, n_permutations=1000, seed=42):
    """
    Generate n_permutations unique strain->binary maps using itertools.
    More memory efficient for large numbers of permutations.
    """
    n_mer = sum(ATCC_GROUPINGS[s] == 0 for s in focused_strains)  # Meropenem
    n_pen = sum(ATCC_GROUPINGS[s] == 5 for s in focused_strains)  # Penicillin
    
    # Create the base label list
    base_labels = [CLASS_TO_BIN[0]] * n_mer + [CLASS_TO_BIN[5]] * n_pen
    
    # Generate all unique permutations
    all_perms = list(itertools.permutations(base_labels))
    
    # Shuffle and take the first n_permutations
    rng = np.random.default_rng(seed)
    rng.shuffle(all_perms)
    
    # Convert to list of dictionaries
    perm_maps = []
    for perm_tuple in all_perms[:n_permutations]:
        perm_map = {s: perm_tuple[i] for i, s in enumerate(focused_strains)}
        perm_maps.append(perm_map)
    
    return perm_maps

In [ ]:
import time
start_time = time.time()

# Generate all unique permutation maps upfront
print(f"\nGenerating {N_PERMUTATIONS} UNIQUE permutations...")
unique_perm_maps = _make_unique_strain_perm_maps(focused_strains, N_PERMUTATIONS, RANDOM_SEED)
print(f"Generated {len(unique_perm_maps)} unique permutations")

# Run permutation test (strain-level mapping applied consistently to both splits)
null_accuracies = []

ref_strains_order = sid_ref_filtered   #  one strain ID per row of X_ref_filtered / y_ref_filtered
te_strains_order  = sid_te_filtered    #  one strain ID per row of X_te_filtered  / y_te_filtered

for perm_idx in range(1, N_PERMUTATIONS + 1):
    # Use the pre-generated unique permutation map
    strain_to_label = unique_perm_maps[perm_idx - 1]
    
    # Apply SAME mapping to BOTH splits, by STRAIN ID per row
    y_ref_permuted = np.array([strain_to_label[s] for s in ref_strains_order], dtype=int)
    y_te_permuted  = np.array([strain_to_label[s] for s in te_strains_order],  dtype=int)

    LAST_PERM_MAP = strain_to_label
    print_permutation_mapping(perm_idx, y_ref_permuted, y_te_permuted)

    # Train & evaluate with permuted labels
    print(f"\nPermutation {perm_idx} training:")
    perm_accuracy = train_eval_permutation_with_epochs(
        X_ref_filtered, y_ref_permuted,
        X_te_filtered,  y_te_permuted,
        epochs=PERM_EPOCHS
    )
    null_accuracies.append(perm_accuracy)
    print(f"Permutation {perm_idx} final accuracy: {perm_accuracy:.2%}")

print("Permutation test completed!")

# End timing
end_time = time.time()
total_time = end_time - start_time
avg_perm_time = total_time / N_PERMUTATIONS

print(f"\n[CELL 13 TIMING]")
print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
print(f"Average time per permutation: {avg_perm_time:.2f} seconds")
print(f"Estimated time for 1000 permutations: {avg_perm_time * 1000 / 60:.2f} minutes")

In [ ]:
# Create histogram visualization for permutation test (Meropenem vs Penicillin)
null_accuracies = np.array(null_accuracies)

plt.figure(figsize=(7, 5))
plt.hist(null_accuracies * 100, bins=20, color='navy', alpha=0.7)

# Observed performance line
plt.axvline(true_accuracy * 100, color='red', linewidth=2, label='True Performance')

# Chance baseline for binary classification = 50%
plt.axvline(50, color='k', linestyle=':', linewidth=1, label='Chance (50%)')

plt.xlabel("Accuracy [%] (on the test set)")
plt.ylabel("Frequency")
plt.title("Permutation null vs observed (Binary: Meropenem vs Penicillin)")

# Add text annotation with observed accuracy and permutation p-value
ymax = max(np.histogram(null_accuracies * 100, bins=20)[0])
p_value = np.mean(null_accuracies >= true_accuracy)

# Position text to avoid legend overlap - move it lower and to the right
plt.text(true_accuracy * 100 + 2, ymax * 0.6,  # Reduced y position from 0.9 to 0.6
         f"Observed = {true_accuracy * 100:.2f}%\nP = {p_value:.4f}",
         fontsize=12, bbox=dict(facecolor='yellow', alpha=0.7))

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()